# Phase 2 - Data Understanding and Data Quality Analysis

This notebook profiles the synthetic Purchase-to-Pay (P2P) event log and case summary datasets. It intentionally does not modify files in `data/raw/`, train machine-learning models, build APIs, dashboards, or GenAI components.

In [ ]:
from pathlib import Path
from collections import Counter

import pandas as pd

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 180)

RAW_DIR = Path("..") / "data" / "raw"
EVENT_LOG_PATH = RAW_DIR / "p2p_event_log.csv"
CASE_SUMMARY_PATH = RAW_DIR / "p2p_case_summary.csv"

## 1. Confirm Raw Dataset Files

Check that both required CSV files exist under `data/raw/`.

In [ ]:
required_files = [EVENT_LOG_PATH, CASE_SUMMARY_PATH]
file_inventory = pd.DataFrame({
    "file": [path.as_posix() for path in required_files],
    "exists": [path.exists() for path in required_files],
    "size_mb": [round(path.stat().st_size / (1024**2), 2) if path.exists() else None for path in required_files],
})
file_inventory

## 2. Load Datasets Without Modifying Raw CSVs

The raw CSVs are read into pandas DataFrames. All cleaning or parsing is done in memory only.

In [ ]:
events = pd.read_csv(EVENT_LOG_PATH)
cases = pd.read_csv(CASE_SUMMARY_PATH)

events.head()

In [ ]:
cases.head()

## 3. Dataset Shape, Schema, Missing Values, Duplicates, and Key Counts

In [ ]:
def profile_dataframe(df: pd.DataFrame, name: str) -> pd.DataFrame:
    return pd.DataFrame({
        "dataset": name,
        "column": df.columns,
        "dtype": [str(dtype) for dtype in df.dtypes],
        "missing_values": df.isna().sum().values,
        "missing_pct": (df.isna().mean().values * 100).round(2),
    })

shape_summary = pd.DataFrame([
    {"dataset": "p2p_event_log.csv", "rows": len(events), "columns": events.shape[1], "duplicate_rows": int(events.duplicated().sum())},
    {"dataset": "p2p_case_summary.csv", "rows": len(cases), "columns": cases.shape[1], "duplicate_rows": int(cases.duplicated().sum())},
])

schema_summary = pd.concat([
    profile_dataframe(events, "p2p_event_log.csv"),
    profile_dataframe(cases, "p2p_case_summary.csv"),
], ignore_index=True)

shape_summary

In [ ]:
schema_summary

In [ ]:
key_counts = pd.DataFrame([
    {"metric": "unique_cases_event_log", "value": events["case_id"].nunique()},
    {"metric": "unique_cases_case_summary", "value": cases["case_id"].nunique()},
    {"metric": "unique_activities", "value": events["activity"].nunique()},
    {"metric": "unique_vendors_event_log", "value": events["vendor_id"].nunique()},
    {"metric": "unique_vendors_case_summary", "value": cases["vendor_id"].nunique()},
    {"metric": "duplicate_event_ids", "value": int(events["event_id"].duplicated().sum())},
    {"metric": "duplicate_case_ids_in_case_summary", "value": int(cases["case_id"].duplicated().sum())},
    {"metric": "case_ids_in_events_not_summary", "value": len(set(events["case_id"]) - set(cases["case_id"]))},
    {"metric": "case_ids_in_summary_not_events", "value": len(set(cases["case_id"]) - set(events["case_id"]))},
])
key_counts

## 4. Date and Time Range

Parse timestamps in memory and check whether any timestamp values fail parsing.

In [ ]:
events["timestamp_dt"] = pd.to_datetime(events["timestamp"], errors="coerce")
cases["start_dt"] = pd.to_datetime(cases["start_time"], errors="coerce")
cases["end_dt"] = pd.to_datetime(cases["end_time"], errors="coerce")

time_summary = pd.DataFrame([
    {"field": "event timestamp", "parse_failures": events["timestamp_dt"].isna().sum(), "min": events["timestamp_dt"].min(), "max": events["timestamp_dt"].max()},
    {"field": "case start_time", "parse_failures": cases["start_dt"].isna().sum(), "min": cases["start_dt"].min(), "max": cases["start_dt"].max()},
    {"field": "case end_time", "parse_failures": cases["end_dt"].isna().sum(), "min": cases["end_dt"].min(), "max": cases["end_dt"].max()},
])
time_summary

## 5. Activity Distribution

In [ ]:
activity_distribution = events["activity"].value_counts().rename_axis("activity").reset_index(name="events")
activity_distribution["pct"] = (activity_distribution["events"] / len(events) * 100).round(2)
activity_distribution

## 6. Priority, Category, Department, and Status Distributions

In [ ]:
def distribution(df: pd.DataFrame, column: str) -> pd.DataFrame:
    result = df[column].value_counts(dropna=False).rename_axis(column).reset_index(name="rows")
    result["pct"] = (result["rows"] / len(df) * 100).round(2)
    return result

priority_distribution = distribution(events, "priority")
category_distribution = distribution(events, "category")
department_distribution = distribution(events, "department")
status_distribution = distribution(events, "status")

priority_distribution

In [ ]:
category_distribution

In [ ]:
department_distribution

In [ ]:
status_distribution

## 7. Validate Chronological Ordering Within Each Case

This validates the current row order in the raw event log for each `case_id`.

In [ ]:
events_with_row_order = events.copy()
events_with_row_order["_row_number"] = range(len(events_with_row_order))

chronological_flags = events_with_row_order.groupby("case_id", sort=False).apply(
    lambda group: group["timestamp_dt"].is_monotonic_increasing,
    include_groups=False,
)
chronological_violations = chronological_flags[~chronological_flags]

pd.DataFrame({
    "chronological_violation_cases": [len(chronological_violations)],
    "total_cases_checked": [events["case_id"].nunique()],
})

In [ ]:
chronological_violations.head(20)

## 8. Identify Process Variants

A process variant is the ordered activity sequence followed by a case. Events are sorted by case, timestamp, and event id before sequence construction.

In [ ]:
events_sorted = events.sort_values(["case_id", "timestamp_dt", "event_id"]).copy()
case_sequences = events_sorted.groupby("case_id")["activity"].agg(tuple)
variant_counts = case_sequences.value_counts()

variants = pd.DataFrame({
    "variant_id": range(1, len(variant_counts) + 1),
    "case_count": variant_counts.values,
    "case_pct": (variant_counts.values / len(case_sequences) * 100).round(2),
    "activity_sequence": [" -> ".join(sequence) for sequence in variant_counts.index],
})
variants.head(20)

## 9. Identify Repeated Activities / Potential Rework

Repeated activities inside a case are treated as potential rework loops, not as duplicate-record issues.

In [ ]:
cases_with_repeated_activities = case_sequences[case_sequences.apply(lambda seq: len(seq) != len(set(seq)))]

repeated_activity_counter = Counter()
for sequence in cases_with_repeated_activities:
    counts = Counter(sequence)
    for activity, count in counts.items():
        if count > 1:
            repeated_activity_counter[activity] += 1

rework_summary = pd.DataFrame([
    {"metric": "cases_with_repeated_activities", "value": len(cases_with_repeated_activities)},
    {"metric": "rework_case_pct", "value": round(len(cases_with_repeated_activities) / len(case_sequences) * 100, 2)},
])

repeated_activity_summary = (
    pd.Series(repeated_activity_counter)
    .sort_values(ascending=False)
    .rename_axis("repeated_activity")
    .reset_index(name="cases")
)

rework_summary

In [ ]:
repeated_activity_summary

## 10. SLA Breach Distribution

In [ ]:
sla_distribution = cases["sla_breached"].value_counts().sort_index().rename_axis("sla_breached").reset_index(name="cases")
sla_distribution["pct"] = (sla_distribution["cases"] / len(cases) * 100).round(2)
sla_distribution

## 11. Compare SLA Breach Rates by Priority, Category, and Vendor

In [ ]:
def breach_rate_by(column: str) -> pd.DataFrame:
    return (
        cases.groupby(column)
        .agg(cases=("case_id", "count"), breaches=("sla_breached", "sum"), breach_rate=("sla_breached", "mean"))
        .assign(breach_rate_pct=lambda df: (df["breach_rate"] * 100).round(2))
        .sort_values(["breach_rate", "cases"], ascending=[False, False])
    )

sla_by_priority = breach_rate_by("priority")
sla_by_category = breach_rate_by("category")
sla_by_vendor = breach_rate_by("vendor_id")

sla_by_priority

In [ ]:
sla_by_category

In [ ]:
sla_by_vendor.head(15)

## 12. Basic Process Duration Statistics

In [ ]:
duration_stats = cases[["duration_hours", "duration_days"]].describe(percentiles=[0.25, 0.5, 0.75, 0.9, 0.95]).T
duration_stats

## 13. Potential Bottleneck Activities from Event-Level Timing

The event log has one timestamp per event, not separate activity start and completion timestamps. Therefore, bottlenecks are approximated using elapsed time between consecutive events inside each case.

In [ ]:
events_sorted["prev_activity"] = events_sorted.groupby("case_id")["activity"].shift(1)
events_sorted["prev_timestamp"] = events_sorted.groupby("case_id")["timestamp_dt"].shift(1)
events_sorted["gap_hours_since_prev"] = (
    events_sorted["timestamp_dt"] - events_sorted["prev_timestamp"]
).dt.total_seconds() / 3600

transition_gaps = (
    events_sorted.dropna(subset=["gap_hours_since_prev"])
    .groupby(["prev_activity", "activity"])
    .agg(
        transitions=("case_id", "count"),
        avg_gap_hours=("gap_hours_since_prev", "mean"),
        median_gap_hours=("gap_hours_since_prev", "median"),
        p90_gap_hours=("gap_hours_since_prev", lambda series: series.quantile(0.9)),
    )
    .round(2)
    .sort_values("avg_gap_hours", ascending=False)
)
transition_gaps.head(15)

In [ ]:
events_sorted["next_timestamp"] = events_sorted.groupby("case_id")["timestamp_dt"].shift(-1)
events_sorted["hours_to_next"] = (
    events_sorted["next_timestamp"] - events_sorted["timestamp_dt"]
).dt.total_seconds() / 3600

activity_waits = (
    events_sorted.dropna(subset=["hours_to_next"])
    .groupby("activity")
    .agg(
        events=("case_id", "count"),
        avg_hours_to_next=("hours_to_next", "mean"),
        median_hours_to_next=("hours_to_next", "median"),
        p90_hours_to_next=("hours_to_next", lambda series: series.quantile(0.9)),
    )
    .round(2)
    .sort_values("avg_hours_to_next", ascending=False)
)
activity_waits

## 14. Integrity Checks and Phase 2 Notes

In [ ]:
event_counts_by_case = events.groupby("case_id").size()
case_event_count_mismatches = (
    cases.set_index("case_id")["event_count"].sort_index() != event_counts_by_case.sort_index()
).sum()

integrity_checks = pd.DataFrame([
    {"check": "event_count_matches_between_files", "issue_count": int(case_event_count_mismatches)},
    {"check": "event_id_duplicates", "issue_count": int(events["event_id"].duplicated().sum())},
    {"check": "case_summary_duplicate_case_id", "issue_count": int(cases["case_id"].duplicated().sum())},
    {"check": "case_ids_in_events_not_summary", "issue_count": len(set(events["case_id"]) - set(cases["case_id"]))},
    {"check": "case_ids_in_summary_not_events", "issue_count": len(set(cases["case_id"]) - set(events["case_id"]))},
])
integrity_checks

## 15. Recommended Next Step

For Phase 3, proceed to process mining and leakage-safe feature engineering. Focus on directly-follows paths, variant analysis, rework loops, SLA breach segmentation, and bottleneck paths before building any machine-learning model.